# Counterfactual analysis

Trying to see that changing what parameters will improve our success chances 

## Loading Model

In [1]:
import joblib

artifacts = joblib.load("../models/lgbm_model_v2.joblib")

lgbm = artifacts["model"]
threshold = artifacts["threshold"]

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/startup_data_final.csv")

x=df.drop(columns=["label","funding_duration_days"])
y=df["label"]

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,stratify=y,random_state=42)

## Select a startup

In [3]:
idx = 90

startup = x_test.iloc[[idx]].copy()
startup

,funding_total_usd,funding_rounds,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,product_crowdfunding,market_mising,...,country_code_ISR,country_code_ITA,country_code_JPN,country_code_NLD,country_code_NOR,country_code_SGP,country_code_SWE,country_code_USA,country_code_missing,country_code_other
3984,1250000.0,2.0,0.0,0.0,0.0,0.0,1250000.0,0.0,0.0,0,...,0,0,0,0,0,0,0,1,0,0


In [4]:
startup["has_institutional_vc"]

3984    0
Name: has_institutional_vc, dtype: int64

## Current prediction

In [5]:
current_fail = 1 - lgbm.predict_proba(startup)[0,1]

print(f"Failure Risk: {current_fail:.2%}")

Failure Risk: 61.27%


## Changing one feature

We know that for our startup "has_institutional_vc"=0 so we are going to change it and some more values to test how much improves so we can suggest the startup owner a solution to save his startup

In [6]:
copystup = startup.copy()


extra_vc_money = 2_000_000
extra_rounds   = 1

copystup["has_institutional_vc"]  = 1
copystup["funding_total_usd"]    += extra_vc_money      # VC money lives inside the total
copystup["funding_rounds"]       += extra_rounds
copystup["funding_stage_reached"] = max(startup["funding_stage_reached"].iloc[0], 2)  # VC-backed => stage >= 2

# derived columns, recompute so the row stays consistent
copystup["funding_per_round"]     = copystup["funding_total_usd"] / copystup["funding_rounds"]
copystup["single_funding_event"]  = int(copystup["funding_rounds"].iloc[0] == 1)

In [7]:
new_fail = 1 - lgbm.predict_proba(copystup)[0,1]

print(f"New Failure Risk: {new_fail:.2%}")

New Failure Risk: 38.50%


## Each level of change brings what

In [8]:
def get_risk(startup_row, vc=False, extra_money=0, extra_rounds=0, bump_stage=False):
    # work on a copy so the original startup never gets touched
    s = startup_row.copy()

    # 1. apply the changes
    if vc:
        s["has_institutional_vc"] = 1

    if bump_stage:
        old_stage = s["funding_stage_reached"].iloc[0]
        s["funding_stage_reached"] = max(old_stage, 2)   # at least stage 2

    s["funding_total_usd"] = s["funding_total_usd"] + extra_money
    s["funding_rounds"]    = s["funding_rounds"] + extra_rounds

    # 2. fix the columns that are calculated from the ones above
    s["funding_per_round"] = s["funding_total_usd"] / s["funding_rounds"]
    s["single_funding_event"] = 1 if s["funding_rounds"].iloc[0] == 1 else 0

    # 3. ask the model (predict_proba gives [fail, success], and index 1 = success)
    success_prob = lgbm.predict_proba(s)[0, 1]
    return 1 - success_prob

In [9]:
print("baseline      ", f"{get_risk(startup):.2%}")
print("VC flag only  ", f"{get_risk(startup, vc=True):.2%}")
print("stage only    ", f"{get_risk(startup, bump_stage=True):.2%}")
print("money only    ", f"{get_risk(startup, extra_money=2_000_000, extra_rounds=1):.2%}")
print("everything    ", f"{get_risk(startup, vc=True, bump_stage=True, extra_money=2_000_000, extra_rounds=1):.2%}")

baseline       61.27%
VC flag only   60.69%
stage only     62.89%
money only     38.11%
everything     38.50%


Tested bumping funding_stage_reached to 2 and risk went up (61.27% → 62.89%). Stage reflects a startup's history rather than a decision a founder can make, so I excluded it from the actionable features.

In [10]:
def get_risk_2(startup_row, vc=False, extra_money=0, extra_rounds=0):
    s = startup_row.copy()

    # apply the changes
    if vc:
        s["has_institutional_vc"] = 1

    s["funding_total_usd"] = s["funding_total_usd"] + extra_money
    s["funding_rounds"]    = s["funding_rounds"] + extra_rounds

    # fix the columns calculated from the ones above
    s["funding_per_round"] = s["funding_total_usd"] / s["funding_rounds"]
    s["single_funding_event"] = 1 if s["funding_rounds"].iloc[0] == 1 else 0

    # ask the model
    return 1 - lgbm.predict_proba(s)[0, 1]

In [11]:
print("baseline      ", f"{get_risk_2(startup):.2%}")
print("VC flag only  ", f"{get_risk_2(startup, vc=True):.2%}")
print("money only    ", f"{get_risk_2(startup, extra_money=2_000_000, extra_rounds=1):.2%}")
print("VC + money    ", f"{get_risk_2(startup, vc=True, extra_money=2_000_000, extra_rounds=1):.2%}")

baseline       61.27%
VC flag only   60.69%
money only     38.11%
VC + money     37.54%


Counterfactual analysis showed that obtaining institutional VC backing and increasing funding can reduce predicted failure risk for many high-risk startups. However, not all features are actionable. Features representing historical outcomes rather than decisions were excluded from recommendations.